# Giai đoạn 1 — Mô hình Custom CNN (Model 1) cho Face Anti-Spoofing

Notebook này thực hiện **Giai đoạn 1** của dự án *Hệ thống CNN Học sâu cho Chống giả mạo Khuôn mặt Đa môi trường*:

- Xây dựng một **CNN tự thiết kế từ đầu** (không dùng backbone pretrained) cho bài toán nhị phân Real/Spoof.
- Huấn luyện & đánh giá trên bộ dữ liệu **LCC-FASD**.
- Chạy **local trên Intel iGPU/XPU** thông qua PyTorch XPU backend.
- Tự động tải dữ liệu từ **Kaggle** nếu chưa có sẵn ở `../dataset/`.
- **Hyperparameter tuning bằng Optuna** (Bayesian optimization / TPE) trước khi train full.
- Đánh giá theo chuẩn **ISO/IEC 30107-3** (APCER / BPCER / ACER).
- Lưu checkpoint **Model 1** để dùng làm ứng viên encoder cho Giai đoạn 2 & 3.

> **Giả định đã thống nhất:**
> - Kiến trúc: CNN kiểu VGG-mini / CDCN-lite (cân bằng tốc độ & độ chính xác).
> - Tuning: Optuna (TPE + Median Pruner).
> - Kaggle: đã có `~/.kaggle/kaggle.json` hoặc biến môi trường `KAGGLE_USERNAME` / `KAGGLE_KEY` sẵn sàng, dùng làm phương án dự phòng nếu `../dataset/` chưa có/thiếu dữ liệu.


## 1. Cài đặt môi trường

Cài PyTorch bản **XPU** (Intel iGPU/Arc, không CUDA) theo đúng lệnh bạn đã kiểm tra hoạt động, cộng thêm các thư viện phụ trợ: `optuna` (hyperparameter tuning), `kaggle` (tải dataset dự phòng), `albumentations` + `opencv-python-headless` (augmentation/đọc ảnh nhanh), `scikit-learn`, `matplotlib`, `seaborn`, `tqdm`, `pandas`.

Nếu bạn đã cài các gói này rồi, có thể bỏ qua cell tiếp theo — cell được viết an toàn để chạy lại nhiều lần (idempotent).


In [1]:
%pip install --upgrade pip
%pip install torch==2.14.0 torchvision --index-url https://download.pytorch.org/whl/xpu
%pip install -q optuna kaggle albumentations opencv-python-headless scikit-learn matplotlib seaborn tqdm pandas pillow

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/xpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Import thư viện & kiểm tra thiết bị XPU/iGPU

Vì bạn xác nhận XPU/iGPU đã chạy ổn định, cell dưới chỉ **kiểm tra và log thông tin thiết bị** (không cài driver/oneAPI). Nếu vì lý do nào đó XPU không sẵn sàng tại thời điểm chạy, code sẽ tự fallback về CPU và cảnh báo rõ ràng thay vì lỗi ngầm.


In [1]:
import os
import io
import json
import time
import random
import shutil
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score

cv2.setNumThreads(0)
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Kiểm tra XPU ---
has_xpu = hasattr(torch, "xpu") and torch.xpu.is_available()
if has_xpu:
    DEVICE = torch.device("xpu")
    n_xpu = torch.xpu.device_count()
    print(f"✅ Đã phát hiện {n_xpu} thiết bị XPU.")
    for i in range(n_xpu):
        try:
            print(f"   - xpu:{i} -> {torch.xpu.get_device_name(i)}")
        except Exception as e:
            print(f"   - xpu:{i} -> (không lấy được tên thiết bị: {e})")
else:
    DEVICE = torch.device("cpu")
    print("⚠️  Không phát hiện XPU khả dụng lúc này -> fallback về CPU.")
    print("    Kiểm tra lại driver Intel GPU / phiên bản torch nếu đây không phải điều bạn mong đợi.")

print(f"\nPyTorch version : {torch.__version__}")
print(f"Thiết bị sử dụng: {DEVICE}")

# bf16 autocast khả dụng tốt trên Intel iGPU/XPU
USE_AMP = has_xpu
AUTOCAST_DEVICE_TYPE = "xpu" if has_xpu else "cpu"
AUTOCAST_DTYPE = torch.bfloat16
print(f"Dùng autocast bf16: {USE_AMP}")


c:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Đã phát hiện 1 thiết bị XPU.
   - xpu:0 -> Intel(R) Graphics

PyTorch version : 2.14.0+xpu
Thiết bị sử dụng: xpu
Dùng autocast bf16: True


## 3. Cấu hình chung

Toàn bộ đường dẫn & hằng số quan trọng được gom vào một chỗ để dễ chỉnh sửa. Mặc định giả sử notebook này nằm trong một thư mục con của repo (ví dụ `notebooks/`), nên dataset ở `../dataset/` theo đúng như bạn mô tả.


In [ ]:
# --- Paths ---
PROJECT_ROOT   = Path("..").resolve()
DATASET_ROOT   = PROJECT_ROOT / "dataset"
MODELS_DIR     = PROJECT_ROOT / "models"
ARTIFACTS_DIR  = PROJECT_ROOT / "artifacts" / "stage1"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_SLUG = "faber24/lcc-fasd"

# --- Image and training ---
IMG_SIZE      = 128
NUM_CLASSES   = 2          # 0 = spoof, 1 = real
LABEL2IDX     = {"spoof": 0, "real": 1}
IDX2LABEL     = {v: k for k, v in LABEL2IDX.items()}

NUM_WORKERS   = 4
CPU_THREADS   = 4
CACHE_IN_RAM  = True
MAX_CACHE_MB  = 6000

torch.set_num_threads(CPU_THREADS)

# --- Optuna hyperparameter tuning ---
N_TRIALS         = 10
TUNE_EPOCHS      = 6
TUNE_TIMEOUT_SEC = None      # e.g. 1800 to cap tuning at 30 minutes
RESUME_TUNING    = True      # persist Optuna study to SQLite and resume missing trials
OPTUNA_STUDY_DB  = ARTIFACTS_DIR / "optuna_stage1.db"

# --- Full training ---
FULL_EPOCHS      = 50
EARLY_STOP_PATIENCE = 10
RESUME_FULL_TRAIN   = True   # load latest checkpoint and continue from the next epoch
MODEL1_CKPT_PATH = MODELS_DIR / "model1_custom_cnn.pt"                 # best checkpoint by val ACER
MODEL1_LATEST_CKPT_PATH = MODELS_DIR / "model1_custom_cnn_latest.pt"   # latest checkpoint for resume

print("Config ready.")
print(f"DATASET_ROOT  = {DATASET_ROOT}")
print(f"MODELS_DIR    = {MODELS_DIR}")
print(f"ARTIFACTS_DIR = {ARTIFACTS_DIR}")
print(f"NUM_WORKERS   = {NUM_WORKERS}")
print(f"CPU_THREADS   = {torch.get_num_threads()}")

Config ready.
DATASET_ROOT  = C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset
MODELS_DIR    = C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\models
ARTIFACTS_DIR = C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\artifacts\stage1
NUM_WORKERS   = 4
CPU_THREADS   = 4


## 4. Check local LCC-FASD, download from Kaggle if missing

This project uses the local layout `../dataset/LCC_FASD/` with three fixed splits:
- `LCC_FASD_training/real|spoof`
- `LCC_FASD_development/real|spoof`
- `LCC_FASD_evaluation/real|spoof`

If that layout is missing or incomplete, the notebook downloads the dataset from Kaggle and checks the same layout again.


In [3]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def split_dirs_for(root: Path):
    return {
        "train": root / "LCC_FASD_training",
        "val":   root / "LCC_FASD_development",
        "test":  root / "LCC_FASD_evaluation",
    }


def count_images(folder: Path) -> int:
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS) if folder.is_dir() else 0


def split_ready(split_dir: Path) -> bool:
    return count_images(split_dir / "real") > 0 and count_images(split_dir / "spoof") > 0


def find_lcc_fasd_root():
    candidates = [DATASET_ROOT / "LCC_FASD", DATASET_ROOT]
    if DATASET_ROOT.exists():
        candidates += [p.parent for p in DATASET_ROOT.rglob("LCC_FASD_training") if p.is_dir()]

    seen = set()
    for root in candidates:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        split_dirs = split_dirs_for(root)
        if all(split_ready(d) for d in split_dirs.values()):
            return root, split_dirs

    root = (DATASET_ROOT / "LCC_FASD").resolve()
    return root, split_dirs_for(root)


LCC_FASD_ROOT, SPLIT_DIRS = find_lcc_fasd_root()
dataset_ready = all(split_ready(d) for d in SPLIT_DIRS.values())

print(f"LCC_FASD_ROOT = {LCC_FASD_ROOT}")
print(f"Dataset ready : {dataset_ready}")


LCC_FASD_ROOT = C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset\LCC_FASD
Dataset ready : True


In [4]:
if not dataset_ready:
    print(f"Local LCC-FASD not found at {LCC_FASD_ROOT} -> downloading from Kaggle...")
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(KAGGLE_DATASET_SLUG, path=str(DATASET_ROOT), unzip=True, quiet=False)
        print("Kaggle download finished.")
    except Exception as e:
        raise RuntimeError(
            "Could not download LCC-FASD from Kaggle automatically. "
            "Check ~/.kaggle/kaggle.json or KAGGLE_USERNAME/KAGGLE_KEY, "
            f"or download '{KAGGLE_DATASET_SLUG}' manually into {DATASET_ROOT}. Original error: {e}"
        )

    LCC_FASD_ROOT, SPLIT_DIRS = find_lcc_fasd_root()
    dataset_ready = all(split_ready(d) for d in SPLIT_DIRS.values())

assert dataset_ready, f"LCC-FASD is still missing or incomplete under {DATASET_ROOT}."
print("Using local dataset splits:")
for name, folder in SPLIT_DIRS.items():
    print(f"  {name:5s}: {folder}")


Using local dataset splits:
  train: C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset\LCC_FASD\LCC_FASD_training
  val  : C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset\LCC_FASD\LCC_FASD_development
  test : C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset\LCC_FASD\LCC_FASD_evaluation


## 5. Build train/val/test sample lists

The local LCC-FASD split is already fixed, so this cell only reads images from `real/` and `spoof/` inside each split directory.


In [5]:
def list_samples(split_dir: Path):
    samples = []
    for label_name in ("spoof", "real"):
        label_idx = LABEL2IDX[label_name]
        label_dir = split_dir / label_name
        samples.extend(
            (str(p), label_idx)
            for p in sorted(label_dir.rglob("*"))
            if p.suffix.lower() in IMG_EXTS
        )
    return samples


train_samples = list_samples(SPLIT_DIRS["train"])
val_samples   = list_samples(SPLIT_DIRS["val"])
test_samples  = list_samples(SPLIT_DIRS["test"])


def summarize(name, samples):
    cnt = Counter(label for _, label in samples)
    print(f"{name:5s}: {len(samples):6d} images | real={cnt.get(1, 0):6d} spoof={cnt.get(0, 0):6d}")


print("\n=== Dataset summary ===")
summarize("train", train_samples)
summarize("val",   val_samples)
summarize("test",  test_samples)

assert train_samples and val_samples and test_samples, "One of train/val/test is empty; please check LCC-FASD paths."



=== Dataset summary ===
train:   8299 images | real=  1223 spoof=  7076
val  :   2948 images | real=   405 spoof=  2543
test :   7580 images | real=   314 spoof=  7266


## 6. Dataset & DataLoader

README ghi rõ: *"data loading, không phải compute, là nút thắt chính"*. Để giảm bottleneck này:

1. Dùng `cv2` để decode ảnh (nhanh hơn PIL thuần trong đa số trường hợp).
2. **Cache trong RAM** ảnh đã decode + resize về đúng `IMG_SIZE` ngay từ lần đọc đầu tiên — các epoch sau không phải decode lại từ đĩa. Có ước lượng dung lượng RAM cần thiết và tự tắt cache nếu vượt `MAX_CACHE_MB`.
3. `num_workers` = số CPU core - 1, để tận dụng đa luồng cho phần augmentation còn lại.


In [6]:
# Ước lượng dung lượng RAM cần cho cache (uint8, HWC, IMG_SIZE x IMG_SIZE x 3)
est_bytes_per_img = IMG_SIZE * IMG_SIZE * 3
total_imgs = len(train_samples) + len(val_samples) + len(test_samples)
est_total_mb = est_bytes_per_img * total_imgs / (1024 ** 2)
print(f"Ước lượng RAM cần nếu cache toàn bộ ({total_imgs} ảnh @ {IMG_SIZE}x{IMG_SIZE}): {est_total_mb:.1f} MB")

if CACHE_IN_RAM and est_total_mb > MAX_CACHE_MB:
    print(f"⚠️  Ước lượng ({est_total_mb:.0f} MB) vượt ngưỡng an toàn ({MAX_CACHE_MB} MB) -> tắt cache RAM.")
    CACHE_IN_RAM = False
else:
    print(f"Cache RAM: {'BẬT' if CACHE_IN_RAM else 'TẮT'}")


class FASDataset(Dataset):
    '''Dataset Real/Spoof đơn giản, hỗ trợ cache ảnh đã decode+resize trong RAM.'''

    def __init__(self, samples, img_size, transform=None, cache_in_ram=False):
        self.samples = samples
        self.img_size = img_size
        self.transform = transform
        self.cache_in_ram = cache_in_ram
        self._cache = {} if cache_in_ram else None

    def __len__(self):
        return len(self.samples)

    def _decode_and_resize(self, path: str):
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            # phòng trường hợp ảnh hỏng -> trả về ảnh xám thay vì crash cả pipeline
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        return img

    def _load(self, idx):
        if self._cache is not None and idx in self._cache:
            return self._cache[idx]
        path, _ = self.samples[idx]
        img = self._decode_and_resize(path)
        if self._cache is not None:
            self._cache[idx] = img
        return img

    def __getitem__(self, idx):
        img = self._load(idx)
        _, label = self.samples[idx]
        if self.transform is not None:
            img = self.transform(image=img)["image"]
        return img, label


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.ImageCompression(quality_lower=60, quality_upper=95),
    ], p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

train_ds = FASDataset(train_samples, IMG_SIZE, transform=train_transform, cache_in_ram=CACHE_IN_RAM)
val_ds   = FASDataset(val_samples,   IMG_SIZE, transform=eval_transform,  cache_in_ram=CACHE_IN_RAM)
test_ds  = FASDataset(test_samples,  IMG_SIZE, transform=eval_transform,  cache_in_ram=CACHE_IN_RAM)

print(f"train_ds={len(train_ds)}  val_ds={len(val_ds)}  test_ds={len(test_ds)}")


Ước lượng RAM cần nếu cache toàn bộ (18827 ảnh @ 128x128): 882.5 MB
Cache RAM: TẮT
train_ds=8299  val_ds=2948  test_ds=7580


In [ ]:
def make_loader(ds, batch_size, shuffle, drop_last=False, num_workers=NUM_WORKERS):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last,
        persistent_workers=num_workers > 0,
        prefetch_factor=4 if num_workers > 0 else None,
    )

# Xem thử một batch để kiểm tra pipeline hoạt động đúng
_tmp_loader = make_loader(train_ds, batch_size=8, shuffle=True)
_imgs, _labels = next(iter(_tmp_loader))
print("Batch shape:", _imgs.shape, "| dtype:", _imgs.dtype, "| labels:", _labels.tolist())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
for i, ax in enumerate(axes.flat):
    img = (_imgs[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(IDX2LABEL[_labels[i].item()])
    ax.axis("off")
plt.suptitle("Ví dụ batch train (đã augment)")
plt.tight_layout()
plt.show()

## 7. Kiến trúc CNN tự thiết kế (Model 1)

Theo lựa chọn của bạn — **cân bằng tốc độ & độ chính xác** — thiết kế theo phong cách **VGG-mini / CDCN-lite**:

- 4 block Conv (mỗi block gồm 2 lớp `Conv3x3 → BatchNorm → ReLU`, sau đó `MaxPool2x2`), số kênh tăng dần `base → 2·base → 4·base → 8·base`.
- Không dùng Dense layer khổng lồ ở cuối: **Global Average Pooling** trước FC head để giảm tham số & chống overfitting (dataset FAS thường không quá lớn).
- `Dropout` ở FC head để tăng khả năng tổng quát hóa.
- `base_channels` và `dropout` là **hyperparameter sẽ được Optuna tối ưu**.

Kiến trúc này lấy cảm hứng từ backbone gọn nhẹ trong các paper CDCN (Central Difference Convolutional Network — Yu et al., 2020) nhưng đơn giản hóa (Conv thường thay vì Central-Difference Conv) để phù hợp vai trò "custom baseline" của Giai đoạn 1, tách biệt rõ với Model 2 (pretrained) và Model 3 (multi-task).


In [10]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CustomFASNet(nn.Module):
    '''CNN tự thiết kế từ đầu cho bài toán Real/Spoof (Model 1).'''

    def __init__(self, in_channels=3, base_channels=32, dropout=0.3, num_classes=NUM_CLASSES):
        super().__init__()
        c = base_channels
        self.features = nn.Sequential(
            ConvBlock(in_channels, c),       # IMG_SIZE   -> IMG_SIZE/2
            ConvBlock(c, c * 2),              # /2         -> /4
            ConvBlock(c * 2, c * 4),          # /4         -> /8
            ConvBlock(c * 4, c * 8),          # /8         -> /16
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(c * 8, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_sanity_model = CustomFASNet()
_sanity_out = _sanity_model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
print("Output shape:", _sanity_out.shape)
print(f"Số tham số: {count_parameters(_sanity_model):,}")
del _sanity_model, _sanity_out


Output shape: torch.Size([2, 2])
Số tham số: 1,206,370


## 8. Hàm đánh giá theo chuẩn ISO/IEC 30107-3

Quy ước nhãn: `1 = real (bona fide)`, `0 = spoof (attack)`.

- **APCER** = tỉ lệ mẫu **spoof** bị phân loại nhầm thành **real** (rủi ro bị qua mặt hệ thống).
- **BPCER** = tỉ lệ mẫu **real** bị phân loại nhầm thành **spoof** (rủi ro từ chối người dùng thật).
- **ACER** = trung bình cộng của APCER và BPCER — chỉ số tổng thể chính, dùng làm mục tiêu tối ưu cho Optuna & early stopping (thay vì Accuracy thông thường, đúng theo mục 6 của README).


In [11]:
def compute_fas_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    spoof_mask = y_true == LABEL2IDX["spoof"]
    real_mask  = y_true == LABEL2IDX["real"]

    apcer = float((y_pred[spoof_mask] == LABEL2IDX["real"]).mean()) if spoof_mask.any() else 0.0
    bpcer = float((y_pred[real_mask] == LABEL2IDX["spoof"]).mean()) if real_mask.any() else 0.0
    acer  = (apcer + bpcer) / 2
    acc   = float((y_pred == y_true).mean())

    metrics = {"accuracy": acc, "APCER": apcer, "BPCER": bpcer, "ACER": acer}
    if y_score is not None:
        try:
            metrics["AUC"] = float(roc_auc_score(y_true, y_score))
        except ValueError:
            metrics["AUC"] = float("nan")
    return metrics


## 9. Vòng lặp train / evaluate

Dùng `torch.autocast` (bf16) trên XPU khi khả dụng để tăng tốc mà không cần `GradScaler` (bf16 không cần scale như fp16).


In [12]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=AUTOCAST_DEVICE_TYPE, dtype=AUTOCAST_DTYPE, enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, total = 0.0, 0
    all_true, all_pred, all_score = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=AUTOCAST_DEVICE_TYPE, dtype=AUTOCAST_DTYPE, enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        probs = F.softmax(outputs.float(), dim=1)[:, LABEL2IDX["real"]]
        preds = outputs.argmax(dim=1)

        running_loss += loss.item() * imgs.size(0)
        total += labels.size(0)
        all_true.extend(labels.cpu().numpy().tolist())
        all_pred.extend(preds.cpu().numpy().tolist())
        all_score.extend(probs.cpu().numpy().tolist())

    metrics = compute_fas_metrics(all_true, all_pred, all_score)
    metrics["loss"] = running_loss / total
    return metrics, all_true, all_pred, all_score


def make_optimizer(model, name, lr, weight_decay):
    name = name.lower()
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True)
    raise ValueError(f"Optimizer không hỗ trợ: {name}")


## 10. Hyperparameter tuning với Optuna

Search space bao gồm: `learning rate`, `weight decay`, `batch size`, `optimizer`, cùng 2 hyperparameter kiến trúc `base_channels` và `dropout`. Mỗi trial chỉ train `TUNE_EPOCHS` epoch ngắn (đủ để so sánh tương đối, không cần hội tụ hoàn toàn), dùng **Median Pruner** để dừng sớm các trial không triển vọng, tiết kiệm thời gian trên máy local.

**Mục tiêu tối ưu: ACER trên tập validation (càng thấp càng tốt)** — nhất quán với chỉ số chính của toàn dự án (mục 6 README), thay vì tối ưu theo Accuracy.


In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial: optuna.Trial) -> float:
    params = {
        "lr":            trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        "weight_decay":  trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size":    trial.suggest_categorical("batch_size", [16, 32, 64]),
        "optimizer":     trial.suggest_categorical("optimizer", ["adam", "adamw", "sgd"]),
        "base_channels": trial.suggest_categorical("base_channels", [16, 24, 32]),
        "dropout":       trial.suggest_float("dropout", 0.1, 0.5),
    }

    train_loader = make_loader(train_ds, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    val_loader   = make_loader(val_ds,   batch_size=params["batch_size"] * 2, shuffle=False)

    model = CustomFASNet(base_channels=params["base_channels"], dropout=params["dropout"]).to(DEVICE)
    optimizer = make_optimizer(model, params["optimizer"], params["lr"], params["weight_decay"])
    criterion = nn.CrossEntropyLoss()

    best_acer = 1.0
    for epoch in range(TUNE_EPOCHS):
        train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_metrics, *_ = evaluate(model, val_loader, criterion, DEVICE)
        best_acer = min(best_acer, val_metrics["ACER"])

        trial.report(val_metrics["ACER"], epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_acer


study_kwargs = {
    "direction": "minimize",
    "sampler": TPESampler(seed=SEED),
    "pruner": MedianPruner(n_warmup_steps=2),
    "study_name": "stage1_custom_cnn_fas",
}
if RESUME_TUNING:
    study_kwargs.update({
        "storage": f"sqlite:///{OPTUNA_STUDY_DB.as_posix()}",
        "load_if_exists": True,
    })

study = optuna.create_study(**study_kwargs)

finished_trials = [t for t in study.trials if t.state in (TrialState.COMPLETE, TrialState.PRUNED, TrialState.FAIL)]
n_finished_trials = len(finished_trials)
remaining_trials = max(0, N_TRIALS - n_finished_trials)
print(f"Start/resume Optuna tuning: finished {n_finished_trials}/{N_TRIALS} trials, running {remaining_trials} more ...")
t0 = time.time()
if remaining_trials > 0:
    study.optimize(objective, n_trials=remaining_trials, timeout=TUNE_TIMEOUT_SEC, show_progress_bar=True)
else:
    print("The saved study already has enough finished trials -> skip tuning rerun.")
t1 = time.time()

completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]
if not completed_trials:
    raise RuntimeError("Optuna has no COMPLETE trial yet, so best_params cannot be selected.")

print(f"\nTuning finished in {(t1 - t0)/60:.1f} minutes for this run.")
print(f"Trials in study: {len(study.trials)} (complete: {len(completed_trials)})")
print(f"Best ACER (val, {TUNE_EPOCHS} epoch): {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  - {k}: {v}")

with open(ARTIFACTS_DIR / "optuna_best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2, ensure_ascii=False)


Start/resume Optuna tuning: finished 0/10 trials, running 10 more ...


  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# Trực quan hóa quá trình tuning (matplotlib backend, không cần plotly/kaleido)
try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
    fig1 = plot_optimization_history(study)
    plt.tight_layout(); plt.show()
    fig2 = plot_param_importances(study)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Không vẽ được biểu đồ Optuna (không ảnh hưởng kết quả tuning): {e}")


## 11. Full training with the best hyperparameters

Use `study.best_params` to train up to `FULL_EPOCHS` epochs with:
- **Early stopping** by validation ACER (`EARLY_STOP_PATIENCE`).
- **Best checkpoint** at `MODEL1_CKPT_PATH` for the lowest validation ACER.
- **Latest/resume checkpoint** at `MODEL1_LATEST_CKPT_PATH` after every epoch, including model, optimizer, scheduler, history, RNG state, and current epoch. If the notebook or machine stops mid-training, rerun the full-train cell and it will continue from the next epoch.
- **Timing benchmark** to compare with the README estimate of 25-40 minutes / 50 epochs.


In [ ]:
best_params = study.best_params
print("Full training with best params:", best_params)

train_loader = make_loader(train_ds, batch_size=best_params["batch_size"], shuffle=True, drop_last=True)
val_loader   = make_loader(val_ds,   batch_size=best_params["batch_size"] * 2, shuffle=False)

model = CustomFASNet(
    base_channels=best_params["base_channels"],
    dropout=best_params["dropout"],
).to(DEVICE)

optimizer = make_optimizer(model, best_params["optimizer"], best_params["lr"], best_params["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FULL_EPOCHS)
criterion = nn.CrossEntropyLoss()

history = defaultdict(list)
best_val_acer = float("inf")
best_epoch = -1
epochs_no_improve = 0
start_epoch = 1


def checkpoint_matches_current_run(ckpt, params):
    return (
        ckpt.get("hyperparameters") == params
        and ckpt.get("config", {}).get("base_channels") == params["base_channels"]
        and ckpt.get("config", {}).get("dropout") == params["dropout"]
        and ckpt.get("config", {}).get("img_size") == IMG_SIZE
        and ckpt.get("config", {}).get("num_classes") == NUM_CLASSES
    )


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if has_xpu:
        try:
            state["xpu"] = torch.xpu.get_rng_state_all()
        except Exception:
            pass
    return state


def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if has_xpu and "xpu" in state:
        try:
            torch.xpu.set_rng_state_all(state["xpu"])
        except Exception as e:
            print(f"Could not restore XPU RNG state; model resume is still valid: {e}")


if RESUME_FULL_TRAIN and MODEL1_LATEST_CKPT_PATH.exists():
    resume_ckpt = torch.load(MODEL1_LATEST_CKPT_PATH, map_location=DEVICE)
    if checkpoint_matches_current_run(resume_ckpt, best_params):
        model.load_state_dict(resume_ckpt["model_state_dict"])
        optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
        history = defaultdict(list, resume_ckpt.get("history", {}))
        best_val_acer = resume_ckpt.get("best_val_acer", best_val_acer)
        best_epoch = resume_ckpt.get("best_epoch", best_epoch)
        epochs_no_improve = resume_ckpt.get("epochs_no_improve", epochs_no_improve)
        start_epoch = int(resume_ckpt["epoch"]) + 1
        restore_rng_state(resume_ckpt.get("rng_state"))
        print(f"Resume full training from {MODEL1_LATEST_CKPT_PATH}: next epoch = {start_epoch}.")
        print(f"Current best: epoch {best_epoch}, val ACER = {best_val_acer:.4f}.")
    else:
        print("Found latest checkpoint, but its hyperparameters/config differ from this run -> ignoring it.")

print(f"\nStart full training: up to {FULL_EPOCHS} epochs on {DEVICE} ...")
t0 = time.time()
last_epoch_trained = start_epoch - 1

if start_epoch > FULL_EPOCHS:
    print(f"Latest checkpoint already reached {last_epoch_trained}/{FULL_EPOCHS} epochs -> no rerun needed.")
elif epochs_no_improve >= EARLY_STOP_PATIENCE:
    print(f"Latest checkpoint already hit early stopping at epoch {last_epoch_trained} -> no rerun needed.")
else:
    for epoch in range(start_epoch, FULL_EPOCHS + 1):
        epoch_t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_metrics, *_ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        epoch_dt = time.time() - epoch_t0
        last_epoch_trained = epoch

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_acer"].append(val_metrics["ACER"])
        history["val_apcer"].append(val_metrics["APCER"])
        history["val_bpcer"].append(val_metrics["BPCER"])

        improved = val_metrics["ACER"] < best_val_acer
        if improved:
            best_val_acer = val_metrics["ACER"]
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "config": {
                    "base_channels": best_params["base_channels"],
                    "dropout": best_params["dropout"],
                    "img_size": IMG_SIZE,
                    "num_classes": NUM_CLASSES,
                    "label2idx": LABEL2IDX,
                },
                "hyperparameters": best_params,
                "val_metrics": val_metrics,
                "epoch": epoch,
            }, MODEL1_CKPT_PATH)
        else:
            epochs_no_improve += 1

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": dict(history),
            "best_val_acer": best_val_acer,
            "best_epoch": best_epoch,
            "epochs_no_improve": epochs_no_improve,
            "config": {
                "base_channels": best_params["base_channels"],
                "dropout": best_params["dropout"],
                "img_size": IMG_SIZE,
                "num_classes": NUM_CLASSES,
                "label2idx": LABEL2IDX,
            },
            "hyperparameters": best_params,
            "last_val_metrics": val_metrics,
            "epoch": epoch,
            "full_epochs": FULL_EPOCHS,
            "rng_state": capture_rng_state(),
        }, MODEL1_LATEST_CKPT_PATH)

        marker = "  <-- best" if improved else ""
        print(f"[{epoch:03d}/{FULL_EPOCHS}] "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
              f"val_ACER={val_metrics['ACER']:.4f} ({epoch_dt:.1f}s){marker}")
        print(f"   Latest checkpoint saved: {MODEL1_LATEST_CKPT_PATH}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no ACER improvement for {EARLY_STOP_PATIENCE} consecutive epochs).")
            break

t1 = time.time()
total_minutes = (t1 - t0) / 60
print(f"\nTrain done. This run took {total_minutes:.1f} minutes; reached epoch {last_epoch_trained} (best epoch = {best_epoch}, val ACER = {best_val_acer:.4f}).")
print(f"   README estimate: ~25-40 minutes for 50 full epochs on local LCC-FASD; use this as a benchmark reference.")
print(f"Best checkpoint saved at: {MODEL1_CKPT_PATH}")
print(f"Latest checkpoint for resume saved at: {MODEL1_LATEST_CKPT_PATH}")


## 12. Biểu đồ quá trình huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()

axes[2].plot(history["val_acer"], label="ACER")
axes[2].plot(history["val_apcer"], label="APCER")
axes[2].plot(history["val_bpcer"], label="BPCER")
axes[2].axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
axes[2].set_title("Val APCER / BPCER / ACER"); axes[2].set_xlabel("epoch"); axes[2].legend()

plt.tight_layout()
fig.savefig(ARTIFACTS_DIR / "training_curves.png", dpi=150)
plt.show()


## 13. Đánh giá cuối cùng trên tập test (evaluation split)

Load lại checkpoint **tốt nhất** (theo ACER trên val) rồi đánh giá trên tập test để có con số cuối cùng của **Model 1**. Lưu ý: đây vẫn là test-split *trong* LCC-FASD (cùng domain) — đánh giá **cross-dataset** thực sự với 4 model sẽ thực hiện ở Giai đoạn 5.


In [ ]:
checkpoint = torch.load(MODEL1_CKPT_PATH, map_location=DEVICE)
best_model = CustomFASNet(
    base_channels=checkpoint["config"]["base_channels"],
    dropout=checkpoint["config"]["dropout"],
).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state_dict"])

test_loader = make_loader(test_ds, batch_size=best_params["batch_size"] * 2, shuffle=False)
test_metrics, y_true, y_pred, y_score = evaluate(best_model, test_loader, nn.CrossEntropyLoss(), DEVICE)

print("=== Kết quả Model 1 trên tập test (in-domain LCC-FASD) ===")
for k, v in test_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[LABEL2IDX['spoof'], LABEL2IDX['real']])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["spoof", "real"], yticklabels=["spoof", "real"])
plt.xlabel("Dự đoán"); plt.ylabel("Thực tế"); plt.title("Confusion matrix — Model 1 (test)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "confusion_matrix_test.png", dpi=150)
plt.show()

with open(ARTIFACTS_DIR / "model1_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)


## 14. Stage 1 summary

- **Model 1 (Custom CNN)** is trained from scratch on LCC-FASD, tuned with Optuna, and evaluated with APCER/BPCER/ACER.
- Best checkpoint: `../models/model1_custom_cnn.pt`, with `state_dict`, architecture config, best hyperparameters, and validation metrics at the best epoch.
- Latest checkpoint for resume: `../models/model1_custom_cnn_latest.pt`, with model, optimizer, scheduler, history, RNG state, and the completed epoch. If training is interrupted, rerun the full-train cell to continue from the next epoch.
- Optuna study: `../artifacts/stage1/optuna_stage1.db`, so tuning can also continue missing trials instead of starting over.
- Other artifacts (best Optuna params, training curves, confusion matrix, test metrics) are saved in `../artifacts/stage1/` for reporting.

**Next step (Stage 2):** train and compare pretrained ResNet50 / EfficientNet-B3 / MobileNetV3-Large backbones on the same LCC-FASD setup, then compare against Model 1 to choose the shared backbone for Stage 3.
